# 🧭 Proyecto final · El Horizonte — Documento del agente
## [Nombre de tu agente]

| | |
|---|---|
| **Integrantes y grupo** | Nombre completo 1 – Grupo · Nombre completo 2 – Grupo · Nombre completo 3 – Grupo |
| **Caso** | [Organización] · ☐ Real ☐ Ficticio |
| **Enlace al video pitch** | [pega aquí el enlace] |
| **Enlace a este cuaderno (solo lectura)** | [Compartir → Cualquier persona con el enlace → Lector] |

> **Cómo usar esta plantilla:** reemplaza todo lo que está entre [corchetes]. Las celdas de texto son tu documento; las celdas de código son la evidencia de que tu agente funciona. Antes de entregar: *Entorno de ejecución → Reiniciar y ejecutar todas*, revisa que no haya errores y guarda el PDF (*Archivo → Imprimir → Guardar como PDF*).

## 1. El caso
[Organización y contexto. ¿Cuál es el problema? ¿A quién afecta? ¿Cómo se resuelve hoy?]

**Línea base:** [un dato o indicador de la situación actual, con su fuente o el supuesto si el caso es ficticio]

## 2. La propuesta
- **Objetivo del agente:** [ ]
- **Usuarios:** [ ]
- **Qué hace:** [ ]
- **Qué NO hace:** [ ]
- **¿Por qué un agente y no otra solución?** [ ]

### Configuración del entorno

In [ ]:
# ── Instalación de librerías ─────────────────────────────────────────────
# El signo "!" ejecuta un comando de la terminal dentro de Colab.
# google-genai : la librería oficial de Google para usar los modelos Gemini.
# pydantic     : sirve para definir "moldes" de datos y pedir respuestas con estructura fija.
# -q (quiet)   : instala sin llenar la pantalla de mensajes.
!pip install -q -U google-genai pydantic

In [ ]:
# ── Configuración: conectar con Gemini ───────────────────────────────────
import time                      # para medir tiempos y esperar entre reintentos
import json                      # para leer y escribir datos en formato JSON
from google import genai         # cliente para hablar con los modelos Gemini
from google.genai import types   # "tipos": objetos de configuración (temperatura, herramientas, etc.)


def obtener_clave():
    """Lee la clave desde los Secretos de Colab; si no existe, la pide por teclado."""
    try:
        from google.colab import userdata
        clave = userdata.get("GOOGLE_API_KEY")
        if clave:
            return clave
    except Exception:
        pass  # no estamos en Colab o el secreto no tiene acceso al cuaderno
    from getpass import getpass  # getpass oculta lo que escribes
    return getpass("Pega tu clave de Gemini y presiona Enter: ")


# El "cliente" es el objeto que envía nuestras preguntas al modelo y trae las respuestas.
client = genai.Client(api_key=obtener_clave())

# Los nombres de los modelos cambian con el tiempo. Probamos una lista en orden de preferencia
# y nos quedamos con el primero que esté disponible para tu clave.
MODELOS_PREFERIDOS = ["gemini-flash-latest", "gemini-2.5-flash", "gemini-2.0-flash"]


def elegir_modelo(preferidos=MODELOS_PREFERIDOS):
    disponibles = []
    for m in client.models.list():
        acciones = m.supported_actions or []
        if "generateContent" in acciones:
            disponibles.append(m.name.replace("models/", ""))
    for nombre in preferidos:
        if nombre in disponibles:
            return nombre
    # Si ninguno de la lista existe, usamos cualquier modelo "flash" disponible.
    flash = [n for n in disponibles if "flash" in n and "image" not in n and "tts" not in n]
    return flash[0] if flash else disponibles[0]


MODELO = elegir_modelo()
print("✅ Conectado. Modelo que usará Nexo:", MODELO)


def llamar_con_reintentos(funcion, *args, intentos=5, **kwargs):
    """Ejecuta una llamada a la API y, si el servicio está saturado (error 429 o 503),
    espera y vuelve a intentar. Así el cuaderno no se detiene por límites de la capa gratuita."""
    espera = 10
    for intento in range(1, intentos + 1):
        try:
            return funcion(*args, **kwargs)
        except Exception as error:
            mensaje = str(error)
            if any(codigo in mensaje for codigo in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE")) and intento < intentos:
                print(f"⏳ Servicio ocupado (intento {intento}). Espero {espera} s y reintento...")
                time.sleep(espera)
                espera *= 2  # cada vez esperamos el doble
            else:
                raise


def preguntar(prompt, sistema=None, temperatura=None, modelo=None, **otros):
    """Envía un texto al modelo y devuelve la respuesta como texto.
    - prompt:      lo que le pedimos al modelo.
    - sistema:     instrucciones de sistema (quién es y cómo se comporta Nexo).
    - temperatura: 0 = respuestas más estables; 1 o más = respuestas más variadas.
    - otros:       cualquier otro ajuste de GenerateContentConfig."""
    config = types.GenerateContentConfig(system_instruction=sistema, temperature=temperatura, **otros)
    respuesta = llamar_con_reintentos(
        client.models.generate_content, model=modelo or MODELO, contents=prompt, config=config
    )
    return respuesta.text


print(preguntar("Saluda en una sola frase, en español."))

### Datos oficiales del caso
[Explica de dónde salen estos datos. Si son ficticios, dilo.]

In [ ]:
# Reemplaza con los datos de tu caso
DATOS = {
    "organizacion": "[nombre]",
}
DOCUMENTOS = [
    "[Regla, política o pregunta frecuente 1]",
    "[Regla, política o pregunta frecuente 2]",
]

## 3. Diseño de prompts (Módulo 1)
**System prompt final y su anatomía** (rol, contexto, instrucción, formato, ejemplos): [explica cada parte]

**Técnicas usadas y por qué:** [few-shot, chain-of-thought, plantillas, patrones…]

**Evolución del prompt:**

| Versión | Qué cambió | Por qué | Resultado |
|---|---|---|---|
| v1 | [ ] | [ ] | [ ] |
| v2 | [ ] | [ ] | [ ] |
| v3 (final) | [ ] | [ ] | [ ] |

In [ ]:
SYSTEM_PROMPT = """[Escribe aquí el system prompt final de tu agente]"""

## 4. Arquitectura del agente (Módulo 2)
[Inserta un diagrama (imagen o texto) y explica: cómo razona y planifica, qué herramientas usa, de dónde saca el conocimiento (RAG, documentos, MCP), qué memoria tiene y qué guardrails limitan su autonomía.]

In [ ]:
# Herramientas del agente: funciones de Python con docstrings claros
def herramienta_ejemplo(parametro: str) -> dict:
    """[Describe qué hace la herramienta y cuándo usarla]"""
    return {"resultado": "[ ]"}

HERRAMIENTAS = [herramienta_ejemplo]

In [ ]:
# Agente con bucle ReAct (puedes copiar y adaptar el de M2_C1)
config = types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, tools=HERRAMIENTAS, temperature=0)

def agente(mensaje):
    r = llamar_con_reintentos(client.models.generate_content, model=MODELO, contents=mensaje, config=config)
    return r.text

print(agente("[Escribe una consulta típica de tu caso]"))

## 5. Supervisión humano-IA (Módulo 3)
- **¿Un agente o un equipo? Patrón de coordinación:** [ ]
- **Roles (agentes y personas):** [ ]
- **Punto de control humano:** [dónde se detiene el agente y quién decide]
- **Protocolo de escalamiento (niveles 1, 2 y 3):** [ ]
- **Métricas y línea base:** [ ]
- **Trazabilidad:** [qué se registra]
- **Alertas:** [umbral → a quién se avisa → qué hacer]
- **Delegación progresiva:** [qué evidencia amplía o reduce la autonomía]

In [ ]:
# Punto de control humano (adáptalo a tu caso)
MODO_INTERACTIVO = False

def pedir_aprobacion(propuesta):
    if MODO_INTERACTIVO:
        return input(f"¿Aprueba esta acción? {propuesta} (s/n): ").strip().lower() == "s"
    return False   # regla simulada para la ejecución completa: explica en el texto cuál usaste

## 6. Construcción y pruebas
**Plataforma y herramientas usadas:** [ ]

**Capturas o enlaces del proceso:** [ ]

In [ ]:
import pandas as pd

# Mínimo 5 casos: incluye uno que deba escalar a una persona y un intento de uso indebido.
CASOS = [
    {"caso": 1, "entrada": "[ ]", "esperado": "[ ]"},
    {"caso": 2, "entrada": "[ ]", "esperado": "[ ]"},
    {"caso": 3, "entrada": "[ ]", "esperado": "[ ]"},
    {"caso": 4, "entrada": "[caso que debe escalar]", "esperado": "Escala a una persona"},
    {"caso": 5, "entrada": "[intento de uso indebido]", "esperado": "Se niega y explica por qué"},
]
for c in CASOS:
    c["obtenido"] = agente(c["entrada"])
pruebas = pd.DataFrame(CASOS)
pruebas["ajuste_realizado"] = "[describe el ajuste que hiciste después de esta prueba]"
pruebas

## 7. Riesgos, ética y límites
[¿A quién puede afectar tu agente y cómo? Alucinaciones, sesgos, privacidad, seguridad. ¿Qué medidas tomaron?]

## 8. Lecciones y próximos pasos
[Qué funcionó, qué no, qué harían distinto y qué necesita el agente para usarse de verdad.]

## 9. Declaración de uso de IA y referencias
**Uso de IA:** [qué herramientas de IA usaron, para qué partes del documento y del video, y cómo verificaron lo que produjeron]

**Referencias (APA 7, mínimo tres fuentes académicas):**
- [ ]
- [ ]
- [ ]